# 03 · Analysis of one take

Masks → spectrogram → band levels relative to baseline → spectra before/after (and the *excess*) →
onset → events and trigger rate → detector check → figures saved.

Everything here reads the cached bundle, so it is fast. Re-run cells freely.

In [ ]:
# ==== YOUR SETTINGS ====
CONFIG = "../config/demo.yaml"
TAKE = "DEMO_002"
BASELINE = (5.0, 34.0)            # [s] quiet reference window BEFORE anything is added
AFTER = (60.0, 170.0)             # [s] window to compare with the baseline
MARKERS = {"drop": 35.0, "acid": 40.0}   # events you noted in the lab book (s from take start)
FMAX = 24000                      # highest frequency to show [Hz]
REF_CHANNEL_FOR_CHIRPS = 4        # the channel that hears the active source best (air mic)
FIND_CHIRPS = False               # True: locate a periodic source automatically and mask it
CHIRP_PERIOD_S = 10.0             # known period from the signal-generator log (None = fit it)

In [ ]:
# ---- standard setup (same in every notebook) ----
%matplotlib inline
import numpy as np, pandas as pd, matplotlib.pyplot as plt
import zoom_acoustics as za
from zoom_acoustics import plots as zp, dsp

cfg = za.load_config(CONFIG)
print("config   :", cfg.config_file)
print("data_dir :", cfg.data_dir)
print("cache_dir:", cfg.cache_dir)
print("figures  :", cfg.figures_dir)

In [ ]:
f = za.load_features(cfg, TAKE)
print(f.take, f"{f.duration_s:.1f} s", "channels", f.channels, "sr", f.sr)
print("masks from config:", len(f.mask_windows), "windows")

## 1. Masks (active chirps, known disturbances)

If a periodic active source was running, every passive number must exclude it. Either put it in the
config (`takes: <take>: periodic: {...}`) **before** processing (best: the detector then ignores it too),
or find it here and add it after the fact. Grey bands in every plot are masked time.

In [ ]:
if FIND_CHIRPS and REF_CHANNEL_FOR_CHIRPS in f.channels:
    g = za.masks.find_periodic_bursts(f.t_level, f.level(REF_CHANNEL_FOR_CHIRPS, "broadband"),
                                      period_s=CHIRP_PERIOD_S)
    print(g)
    if g:
        # the chirp sweep starts at 100 Hz; pad generously before the detected rise
        f.add_masks(za.masks.periodic_windows(g["first_s"], g["period_s"], f.duration_s,
                                              pad_before_s=0.5, pad_after_s=1.5))
live = za.masks.live_seconds(0, f.duration_s, f.mask_windows)
print(f"masked: {100*(1-live/f.duration_s):.1f} % of the take")

## 2. Spectrogram

In [ ]:
fig = zp.plot_spectrogram(f, fmax=FMAX, markers=MARKERS)
zp.save(fig, cfg, f"{TAKE}_spectrogram")

## 3. Band levels, as change relative to the baseline

Absolute dB values are **uncalibrated** (dB re digital full scale) and differ between sensors for
reasons unrelated to the sample. The change relative to a quiet baseline, within one channel, is the robust quantity.

In [ ]:
bands = [b for b in ("low", "audio", "rig", "hb1", "hb2", "hb3") if b in f.bands]
fig = zp.plot_levels(f, bands=bands, dt=1.0, markers=MARKERS, relative_to=BASELINE)
zp.save(fig, cfg, f"{TAKE}_levels")

## 4. Spectra: baseline vs after, and the excess

The dotted line is `after − baseline` in **linear power**: the spectrum of what was *added*. Grey = the
`rig` band (a 5.6 kHz apparatus resonance in the Sep-2026 setup; check whether yours has one in a
water-only take).

In [ ]:
fig = zp.plot_psd(f, {"baseline": BASELINE, "after": AFTER}, fmax=FMAX, fmin=100, excess=True)
zp.save(fig, cfg, f"{TAKE}_spectra")

## 5. Onset

First time the audio-band level rises 6 dB above baseline **and stays up** for 10 s. The hold
requirement is what rejects a single knock or the sample being dropped in.

In [ ]:
rows = []
for c in f.channels:
    on, base = f.onset(c, "audio", base_window=BASELINE, rise_db=6, hold_s=10)
    rows.append(dict(channel=c, label=f.label(c), onset_s=on, baseline_dB=dsp.db(base)))
onsets = pd.DataFrame(rows); onsets

## 6. Events (STA/LTA triggers) and trigger rate

A trigger is a short transient that stands out from the preceding 60 ms. **A trigger rate is not a
bubble rate:** at high bubble rates triggers merge and the LTA is inflated, so the rate saturates
(see `docs/VALIDATION.md`, R1/R2). Compare rates within a channel, over time.

In [ ]:
det = f.meta["detect_bands"][0] if f.meta["detect_bands"] else None
if det:
    fig = zp.plot_event_rate(f, det, bin_s=5.0, markers=MARKERS)
    zp.save(fig, cfg, f"{TAKE}_trigger_rate")

### Check the detector by eye

Before trusting any count, look at what the detector saw in a short window: envelope, STA/LTA ratio,
thresholds and the triggers. Pick a window inside the reaction.

In [ ]:
if det and f.meta["env_stored"]:
    ch0 = f.channels[0]
    t_chk = (AFTER[0], AFTER[0] + 0.5)
    fig = zp.plot_detector(f, ch0, det, *t_chk)

### Re-run the detector with other settings (no need to re-process)

Only possible when the fine envelope was stored (`env_stored` is True).

In [ ]:
if det and f.meta["env_stored"]:
    for on in (5, 8, 12):
        ev, live = f.redetect(f.channels[0], det, on=on)
        print(f"on={on:>2}: {len(ev)} triggers, {len(ev)/live:.1f} per live second")

## 7. Summary table for this take

In [ ]:
tab = za.timeline.window_table([f], {"baseline": BASELINE, "after": AFTER}, bands=bands)
tab.pivot_table(index=["channel", "label"], columns=["band", "window"], values="level_dB").round(1)

In [ ]:
tab[tab["window"] == "after"].pivot_table(index=["channel", "label"], columns="band", values="delta_dB").round(1)